# 03 — Mutual Information

Measure how much two variables share — the bits one carries about the other. Build it as the relative entropy of `02/02` pointed at a joint table and its own product of marginals, compute it by hand on the three tables of `02/01` §3, derive its three faces ($D$ of the joint, a difference of entropies, a saved conditional entropy), prove the ceiling that two bits can never share more than one bit (the ceiling a Bell pair will clear in `02/10`), meet the conditional version and the data-processing inequality that `02/04` is built on, and learn why a mutual information *estimated from data* is biased upward — the first honesty check of every coupling analysis.

**Prerequisites:** `02/01_shannon_entropy` (joint and conditional entropy, the chain rule, the three tables of §3, exercise 3's subadditivity gap), `02/02_kl_divergence` (relative entropy, Gibbs, the chain rule of $D$, the data-processing inequality, $D$ on a flattened table), `01/16_tensor_and_partial_trace` (the independent table, the correlated coins, $\langle Z\otimes Z\rangle$ vs $\langle Z\rangle\langle Z\rangle$ as the seed of "correlation"), `01/19_quantum_channels` (a stochastic matrix as a noisy channel).

**What you'll be able to do**
- Define $I(X;Y) = D(p_{XY}\,\|\,p_X \otimes p_Y)$, compute it term by term on the $90\%$-agreement table, and derive by hand $I = H(X) + H(Y) - H(XY) = H(X) - H(X|Y)$: the bits saved by describing the pair together.
- Prove $I \ge 0$ with equality iff independent (Gibbs), symmetry, and the ceiling $I \le \min(H(X), H(Y))$ with equality iff one variable is a function of the other — so two bits share at most $1$ bit.
- Read $I$ as what survives a noisy channel: $1 - h(\epsilon)$ for a bit through a symmetric flip with probability $\epsilon$, and see that $I$ is blind to the sign of a relationship and alive to nonlinear ones.
- Define conditional mutual information $I(X;Y|Z)$, derive the chain rule $I(X;YZ) = I(X;Y) + I(X;Z|Y)$ and from it the data-processing inequality $I(X;Z) \le I(X;Y)$ for $X \to Y \to Z$ — the objects `02/04` turns into transfer entropy.
- Measure the upward bias of the plug-in estimator on independent data, match it to $\frac{(|X|-1)(|Y|-1)}{2N\ln 2}$, and state the rule: compare against surrogates.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import conditional_mutual_information, mutual_information, shannon_entropy

rng = np.random.default_rng(0)   # random channels (section 3) and finite samples (section 4)
viz.use_course_style()

def h(p):
    return shannon_entropy([p, 1 - p])   # binary entropy of 02/01

## 1. Shared bits: relative entropy of a table to its own product

`02/02` ended by applying $D$ to flattened tables. Point it at the most natural pair: a joint table $p_{XY}$ and the product of its own marginals $p_X \otimes p_Y$ — the table the pair *would* have if $X$ and $Y$ were independent (`01/16` §4). The distance from the truth to that independent impostor is the **mutual information**

$$
I(X;Y) = D\big(p_{XY}\,\big\|\,p_X \otimes p_Y\big) = \sum_{x,y} p(x,y)\log_2\frac{p(x,y)}{p(x)\,p(y)} .
$$

Two facts are immediate from `02/02`. **Gibbs:** $I \ge 0$, with equality if and only if $p_{XY} = p_X \otimes p_Y$ — independence. **Symmetry:** the formula does not care which variable is called $X$ (transpose the table, nothing changes) — unlike $D$ itself, whose two arguments play different roles; here the second argument is *built from* the first. Now expand the logarithm, by hand, using $\sum_y p(x,y) = p(x)$ and $\sum_x p(x,y) = p(y)$:

$$
I(X;Y) = \underbrace{-\sum_{x,y} p(x,y)\log_2 p(x)}_{H(X)} \;\underbrace{-\sum_{x,y} p(x,y)\log_2 p(y)}_{H(Y)} \;\underbrace{+\sum_{x,y} p(x,y)\log_2 p(x,y)}_{-H(XY)} = H(X) + H(Y) - H(XY),
$$

the **subadditivity gap** you met in `02/01` exercise 3 — the bits you save by describing the pair together rather than separately. And with the chain rule $H(XY) = H(Y) + H(X|Y)$ of `02/01` §3:

$$
I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X),
$$

the uncertainty about $X$ that learning $Y$ removes — and, by symmetry, the same number the other way round. Three faces, one quantity — and a fourth, from the chain rule of $D$ in `02/02` §4(a): $I(X;Y) = \sum_x p(x)\,D(p_{Y|x}\,\|\,p_Y)$, how much learning $x$ moves the distribution of $y$, on average — the per-value view that section 3 will condition on a third variable. (`02/01` called the pair $A, B$; from here on $X, Y$, Cover & Thomas's convention, with $A{:}B$ kept for the quantum side.) Let's compute it term by term on the $90\%$-agreement table of `02/01`, then on its two neighbours (the independent table and the correlated coins of `01/16`), against the helper `mutual_information`.


In [ ]:
def mi_by_hand(P):
    P = np.asarray(P, dtype=float)
    pX, pY = P.sum(axis=1), P.sum(axis=0)
    prod = np.outer(pX, pY)                    # the independent impostor
    nz = P > 0
    return float((P[nz] * np.log2(P[nz] / prod[nz])).sum()) + 0.0

def H(p):
    return shannon_entropy(np.asarray(p).ravel())

tables = {
    "agree 90%":                        np.array([[0.45, 0.05], [0.05, 0.45]]),
    "independent (0.3,0.7)x(0.4,0.6)": np.outer([0.3, 0.7], [0.4, 0.6]),
    "correlated coins (01/16 s.5)":     np.array([[0.5, 0.0], [0.0, 0.5]]),
    "anti-correlated coins":            np.array([[0.0, 0.5], [0.5, 0.0]]),
}

A = tables["agree 90%"]
pX, pY = A.sum(axis=1), A.sum(axis=0)
print("the 90% table, term by term")
print("  joint p(x,y):\n", A)
print("  product p(x)p(y):\n", np.outer(pX, pY))
print("  p(x,y) log2 [p(x,y) / p(x)p(y)]:\n", np.round(A * np.log2(A / np.outer(pX, pY)), 4))
print(f"  sum = {mi_by_hand(A):.4f} bits;  H(X)+H(Y)-H(XY) = {H(pX) + H(pY) - H(A):.4f};  H(X)-H(X|Y) = {H(pX) - (H(A) - H(pY)):.4f};  1 - h(0.1) = {1 - h(0.1):.4f}")

print(f"\n{'table':34s} {'I by hand':>10s} {'helper':>8s} {'H(X)':>7s} {'H(Y)':>7s} {'H(XY)':>7s}   I == I^T")
for name, P in tables.items():
    print(f"{name:34s} {mi_by_hand(P):10.4f} {mutual_information(P):8.4f} {H(P.sum(1)):7.4f} {H(P.sum(0)):7.4f} {H(P):7.4f}   {np.isclose(mi_by_hand(P), mi_by_hand(P.T))}")

**Read the output.** On the $90\%$ table the two diagonal terms are positive ($0.45$ where independence would put $0.25$: the pair agrees more than chance) and the two off-diagonal terms negative (disagreement happens *less* than chance); the sum is $0.5310$ bits, and all three faces give the same number — including the closed form $1 - h(0.1)$: agreeing $90\%$ of the time leaves $h(0.1) = 0.469$ bits of uncertainty about $Y$ once $X$ is known, so $1 - 0.469$ bits are shared. The independent table shares $0$ to machine precision (the table *is* its product; Gibbs' equality case). The correlated coins share $1$ bit — everything: $H(X) = H(Y) = H(XY) = 1$, knowing one fixes the other. And the *anti*-correlated coins share $1$ bit too: mutual information is blind to the sign of a relationship — it asks whether the table factorises, not which way the diagonal leans. (That is a feature: in `02/04` and module 04 the relationships you measure will not be linear.) Every table gives the same $I$ transposed.


## 2. The ceiling, and what a noisy channel lets through

**The ceiling.** From $I = H(X) - H(X|Y)$ and $H(X|Y) \ge 0$ (`02/01` §3): $I(X;Y) \le H(X)$, and by symmetry $I \le H(Y)$, so

$$
0 \le I(X;Y) \le \min\big(H(X), H(Y)\big),
$$

with equality on the right iff $H(X|Y) = 0$ — $X$ is a *function* of $Y$ (or the other way). For two *bits* that is $I \le 1$: the coins sit exactly on the ceiling, and no classical pair of bits can share more. Keep that number: in `02/10` a Bell pair will share $2$ bits, and the only step this proof uses that quantum mechanics breaks is $H(X|Y) \ge 0$ — the Bell state's conditional entropy is $-1$ (`01/18`, `02/11`).

**A noisy channel.** Here is the cleanest way to see $I$ move. Send a fair bit $X$ through the **binary symmetric channel** — the stochastic matrix of `01/19` that flips the bit with probability $\epsilon$ — and call the output $Y$. The joint table is $p(x, y) = \tfrac12\,T_{yx}$, the output is still a fair bit, $H(Y) = 1$, and the residual uncertainty about $Y$ given $X$ is the flip coin: $H(Y|X) = h(\epsilon)$. So

$$
I(X;Y) = H(Y) - H(Y|X) = 1 - h(\epsilon):
$$

$1$ bit at $\epsilon = 0$ (the coins), $0$ at $\epsilon = \tfrac12$ (the output is a fresh coin, the input is forgotten), and — note — $1$ bit again at $\epsilon = 1$: a channel that *always* flips is perfectly informative, sign-blindness once more. Shannon (1948) showed this is the most any code can push through that channel per use — its *capacity* — and that is the historical reason mutual information exists at all. Let's draw $I$ against $\epsilon$ on top of $1 - h(\epsilon)$, and next to it the blend from the independent table to the coins, $p_t = (1 - t)\,p_X\otimes p_Y + t\,p_{\text{coins}}$, which is the same curve in disguise: the blended table agrees with probability $(1 + t)/2$, so $I = 1 - h\big(\tfrac{1+t}{2}\big)$.


In [ ]:
eps = np.linspace(0.0, 1.0, 201)
I_bsc = []
for e in eps:
    T = np.array([[1 - e, e], [e, 1 - e]])          # T[y, x]: columns are probability vectors
    joint = 0.5 * T.T                               # p(x, y) = p(x) T[y, x] with p(x) = 1/2
    I_bsc.append(mi_by_hand(joint))
I_bsc = np.array(I_bsc)

ts = np.linspace(0.0, 1.0, 101)
independent = np.outer([0.5, 0.5], [0.5, 0.5])
coins = tables["correlated coins (01/16 s.5)"]
I_blend = np.array([mi_by_hand((1 - t) * independent + t * coins) for t in ts])

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot(eps, I_bsc, color=viz.SOURCE_COLOR, lw=2.4, label="I(X;Y) by hand")
ax.plot(eps, 1 - np.array([h(e) for e in eps]), color=viz.TARGET_COLOR, lw=1.2, ls="--", label="1 - h(eps)")
ax.axhline(1.0, color=COLORS["highlight"], lw=1, ls=":", label="ceiling min(H(X), H(Y)) = 1 bit")
ax.set_xlabel("flip probability eps"); ax.set_ylabel("mutual information (bits)")
ax.set_title("A fair bit through a symmetric flip channel", pad=10); ax.legend(loc="center")
ax = axes[1]
ax.plot(ts, I_blend, color=viz.SOURCE_COLOR, lw=2.4, label="I by hand on the blended table")
ax.plot(ts, 1 - np.array([h((1 + t) / 2) for t in ts]), color=viz.TARGET_COLOR, lw=1.2, ls="--", label="1 - h((1+t)/2)")
ax.axhline(1.0, color=COLORS["highlight"], lw=1, ls=":")
ax.set_xlabel("t   (0 = independent table, 1 = correlated coins)"); ax.set_ylabel("mutual information (bits)")
ax.set_title("Blending independence into the coins", pad=10); ax.legend(loc="center left")
plt.show()

print(f"eps = 0: {I_bsc[0]:.4f}   eps = 0.1: {I_bsc[20]:.4f} (= the 90% table)   eps = 0.5: {I_bsc[100]:.4f}   eps = 1: {I_bsc[-1]:.4f}")
print(f"max |I - (1 - h(eps))| = {np.max(np.abs(I_bsc - (1 - np.array([h(e) for e in eps])))):.2e}")
# the ceiling on 2000 random 2x2 tables, and a 3x2 table where Y is a function of X
rand = rng.dirichlet(np.ones(4), size=2000).reshape(-1, 2, 2)
slack = np.array([min(H(P.sum(1)), H(P.sum(0))) - mi_by_hand(P) for P in rand])
print(f"2000 random 2x2 tables: min over tables of  min(H(X),H(Y)) - I  = {slack.min():.4f}  (never negative)")
F = np.array([[1 / 3, 0.0], [0.0, 1 / 3], [1 / 3, 0.0]])     # Y = (X == 1): a function of X, 3 x 2
print(f"a 3x2 table where Y is a function of X: I = {mi_by_hand(F):.4f} = H(Y) = {H(F.sum(0)):.4f}  <  H(X) = {H(F.sum(1)):.4f}  -- the smaller entropy is the ceiling, and it is reached")

**Read the figure and the output.** Left: the by-hand mutual information of a fair bit through the flip channel lies exactly on $1 - h(\epsilon)$ (difference $10^{-16}$): $1$ bit at $\epsilon = 0$, $0.531$ at $\epsilon = 0.1$ — the $90\%$ table *is* this channel — $0$ at $\epsilon = \tfrac12$, and back to $1$ at $\epsilon = 1$, the always-flip channel: a symmetric U. Right: blending the independent table into the coins traces the right half of the same U, since the blend agrees with probability $(1 + t)/2$; at $t = 0$ nothing is shared, at $t = 1$ everything. The rose dotted line is the ceiling: two thousand random two-bit tables never cross it (the smallest slack is positive), and a $3\times 2$ table in which $Y$ is a deterministic function of $X$ reaches it exactly — $I = H(Y) = 0.918$ bits, the *smaller* of the two entropies, while $H(X) = \log_2 3$ stays out of reach: a function of $X$ can carry at most its own entropy.


## 3. Conditioning, the chain rule, and the data-processing inequality

Transfer entropy (`02/04`) asks a three-variable question — what does the past of $Y$ tell me about the future of $X$ *beyond what the past of $X$ already tells me?* — so you need mutual information **given** a third variable. Define it as the average, over $z$, of the mutual information of the conditional table $p(x, y\,|\,z)$:

$$
I(X;Y\,|\,Z) = \sum_z p(z)\, I\big(X;Y\,\big|\,Z = z\big) = \sum_{x,y,z} p(x,y,z)\log_2\frac{p(x,y|z)}{p(x|z)\,p(y|z)} = H(XZ) + H(YZ) - H(Z) - H(XYZ),
$$

the last form by the same log-expansion as section 1 — write $\log p(x,y|z) = \log p(x,y,z) - \log p(z)$ and $\log p(x|z)\,p(y|z) = \log p(x,z) + \log p(y,z) - 2\log p(z)$, and each term averages to an entropy (this is what the helper `conditional_mutual_information` computes; its axes are $(X, Y, Z)$ with $Z$ the conditioner). It is $\ge 0$ — an average of mutual informations — and $= 0$ iff $X$ and $Y$ are independent once $Z$ is known: the **Markov** property. Now the **chain rule**, by hand from the entropy forms: $I(X;YZ) = H(X) + H(YZ) - H(XYZ)$; add and subtract $H(Y)$ and $H(XY)$ and regroup,
$$
I(X;YZ) = \underbrace{H(X) + H(Y) - H(XY)}_{I(X;Y)} + \underbrace{H(XY) + H(YZ) - H(Y) - H(XYZ)}_{I(X;Z|Y)} ,
$$
and, by the symmetry of the left side in $Y$ and $Z$, also $I(X;YZ) = I(X;Z) + I(X;Y|Z)$ — the two ways of learning $Y$ and $Z$ one after the other.

**Data processing.** Let $Z$ be obtained from $Y$ alone — a noisy channel, a summary, a rounding: $X \to Y \to Z$, so $I(X;Z|Y) = 0$. Then the two chain-rule expansions give $I(X;Y) = I(X;YZ) = I(X;Z) + I(X;Y|Z) \ge I(X;Z)$:

$$
\boxed{\,I(X;Z) \le I(X;Y)\,}\qquad\text{for any chain } X \to Y \to Z .
$$

No processing of $Y$ can *increase* what it tells you about $X$ — `02/02` §4(c), now for mutual information (it is the same theorem: push $p_{XY}$ and $p_X \otimes p_Y$ through $\mathrm{id}\otimes T$). This is the inequality that makes "a symmetric measure cannot say who drives whom" more than a slogan in `02/04`, and the one the quantum mutual information also obeys under local channels (a fact `02/10` will state). Let's build a chain from the $90\%$ table through a second channel, check the chain rule both ways by hand, and test the inequality against two thousand random channels.


In [ ]:
def joint3(PXY, T):
    # X -> Y -> Z: p(x, y, z) = p(x, y) T[z, y]; axes (X, Y, Z)
    return np.einsum("xy,zy->xyz", PXY, T)

def cmi_by_hand(P):            # I(X;Y|Z) on axes (X, Y, Z), via the four-entropy form; H flattens any table
    P = np.asarray(P, dtype=float)
    return float(H(P.sum(1)) + H(P.sum(0)) - H(P.sum((0, 1))) - H(P)) + 0.0

T2 = np.array([[0.8, 0.3], [0.2, 0.7]])          # a second, asymmetric channel: T2[z, y], columns sum to 1
P3 = joint3(A, T2)
I_XY = mi_by_hand(A); I_XZ = mi_by_hand(P3.sum(axis=1)); I_XYZ = mi_by_hand(P3.reshape(2, 4))
I_XZ_given_Y = cmi_by_hand(P3.transpose(0, 2, 1))  # condition on Y: move Y to the last axis
I_XY_given_Z = cmi_by_hand(P3)                      # condition on Z
print(f"I(X;Z|Y) = {I_XZ_given_Y:.2e}   (Markov: Z was made from Y alone)      helper: {conditional_mutual_information(P3.transpose(0, 2, 1)):.2e}")
print(f"I(X;Y|Z) = {I_XY_given_Z:.4f}                                              helper: {conditional_mutual_information(P3):.4f}")
print(f"chain rule 1: I(X;YZ) = {I_XYZ:.4f}  =  I(X;Y) + I(X;Z|Y) = {I_XY:.4f} + {I_XZ_given_Y:.4f} = {I_XY + I_XZ_given_Y:.4f}")
print(f"chain rule 2: I(X;YZ) = {I_XYZ:.4f}  =  I(X;Z) + I(X;Y|Z) = {I_XZ:.4f} + {I_XY_given_Z:.4f} = {I_XZ + I_XY_given_Z:.4f}")
print(f"data processing: I(X;Z) = {I_XZ:.4f}  <=  I(X;Y) = {I_XY:.4f}   (the second channel threw away {I_XY - I_XZ:.4f} bits)")

worst = -np.inf
for _ in range(2000):
    T = rng.dirichlet(np.ones(2), size=2).T      # random 2x2 channel, columns sum to 1
    worst = max(worst, mi_by_hand(joint3(A, T).sum(axis=1)) - I_XY)
print(f"2000 random channels after the 90% table: max of  I(X;Z) - I(X;Y)  = {worst:.3e}  (never positive)")
perm = np.array([[0.0, 1.0], [1.0, 0.0]])
print(f"a reversible channel (the flip): I(X;Z) = {mi_by_hand(joint3(A, perm).sum(axis=1)):.4f} = I(X;Y)  -- nothing lost, so I(X;Y|Z) = 0 and equality holds")

**Read the output.** With $Z$ made from $Y$ alone, $I(X;Z|Y)$ is zero to $10^{-16}$ — the Markov property — and both chain-rule expansions land on the same $I(X;YZ) = 0.531$ bits: once through $I(X;Y) + 0$, once through $I(X;Z) + I(X;Y|Z) = 0.120 + 0.411$. Read the second one as a budget: of the $0.531$ bits $Y$ carries about $X$, only $0.120$ survive the second channel, and $0.411$ remain visible only if you still hold $Y$. Two thousand random channels never raise $I(X;Z)$ above $I(X;Y)$, and the reversible flip loses nothing. The equality case follows from the derivation: $I(X;Z) = I(X;Y)$ iff $I(X;Y|Z) = 0$ — $Z$ already holds everything $Y$ knew about $X$ (a *sufficient statistic*; the quantum version of this idea returns in `02/09`'s lift) — which is automatic when $Y$ can be recovered from $Z$, as for the flip, but does not require it. This is the asymmetry `02/04` exploits: conditioning on a variable's own past and asking what *another* variable's past adds is $I(X_{t+1}; Y_t\,|\,X_t)$ — today's conditional mutual information, with the axes named by time.


## 4. Mutual information from data: the plug-in estimator is biased upward

In every application of this course — module 04's coupling measures, the hyperscanning questions behind them — you will not be handed $p(x, y)$. You will count: $N$ paired samples, a table of frequencies $\hat p(x,y)$, and $I(\hat p)$, the **plug-in** estimate. Here is the trap. Even for *independent* variables, a finite sample never factorises exactly — the counts fluctuate — and since $I \ge 0$ with equality only at exact factorisation, every fluctuation pushes the estimate *up*. The bias is not a subtlety of small samples; it is a bias of sign, and its leading term is known (Miller 1955; the $\chi^2$ limit of the log-likelihood ratio, $2N\ln 2\, \hat I \to \chi^2$ with $(|X|-1)(|Y|-1)$ degrees of freedom):

$$
\mathbb E\big[\hat I\big] - I \;\approx\; \frac{(|X| - 1)(|Y| - 1)}{2N\ln 2}\ \text{bits}
$$

(leading order in $1/N$, for a well-populated table; subtracting it is the **Miller–Madow correction**). For two bits and $N = 100$ that is $0.0072$ bits — small, but it is *pure artefact*, and it grows with the alphabet size (binning a continuous signal into $8$ levels: $49/(2N\ln2)$). The discipline, which `02/04` repeats for transfer entropy: **never read an estimated $I$ against zero — read it against the distribution of $\hat I$ on surrogate data** (shuffled pairings, which destroy the dependence and keep the marginals), and subtract or test. Let's measure the bias on independent fair bits for four sample sizes and compare with the formula, then watch the same $N = 100$ estimator on the $90\%$ table.


In [ ]:
def plug_in_mi(x, y, k=2):
    counts = np.zeros((k, k)); np.add.at(counts, (x, y), 1.0)
    return mi_by_hand(counts / counts.sum())

trials = 2000
print(f"{'N':>6s} {'mean I_hat (indep.)':>20s} {'(k-1)^2/(2N ln2)':>18s} {'95th pct':>9s}")
for N in [50, 100, 400, 1600]:
    ests = np.array([plug_in_mi(rng.integers(0, 2, N), rng.integers(0, 2, N)) for _ in range(trials)])
    print(f"{N:6d} {ests.mean():20.5f} {1 / (2 * N * np.log(2)):18.5f} {np.percentile(ests, 95):9.5f}")

# the same estimator on the 90% table, N = 100, against a surrogate (shuffled) null
N = 100
xy = rng.choice(4, size=(trials, N), p=A.ravel())      # 0..3 encodes (x, y)
x, y = xy // 2, xy % 2
est = np.array([plug_in_mi(x[i], y[i]) for i in range(trials)])
null = np.array([plug_in_mi(x[i], rng.permutation(y[i])) for i in range(trials)])
print(f"\n90% table, N = {N}: true I = {I_XY:.4f};  plug-in mean = {est.mean():.4f} (bias {est.mean() - I_XY:+.4f});  shuffled-pairs null: mean {null.mean():.4f}, 95th pct {np.percentile(null, 95):.4f}")
print(f"   share of estimates above the null's 95th percentile: {np.mean(est > np.percentile(null, 95)):.3f}")

**Read the output.** On *independent* bits the plug-in estimate is never zero on average: $0.0146$ bits at $N = 50$, $0.0069$ at $N = 100$, down to $0.00043$ at $N = 1600$ — each within a few percent of $\frac{1}{2N\ln 2}$, and the $95$th percentile is three to four times the mean. The bias shrinks as $1/N$, it does not vanish, and it is always positive. On the $90\%$ table with $N = 100$ the estimate is biased by a similar $+0.006$ bits above the true $0.531$ — harmless there — but the point is the comparison: the shuffled-pairs null sits near $0.007$ with a $95$th percentile near $0.03$, and essentially every estimate clears it. That is the honest test, and it is the only one you will use from `02/04` onward: an estimated mutual information is a number *relative to its surrogate distribution*, never relative to zero.


## 5. Where it goes

- **Transfer entropy** (`02/04`): $\mathrm{TE}_{Y\to X} = I(X_{t+1};\,Y_t\,|\,X_t)$ — today's conditional mutual information with the axes named by time. Symmetric $I$ cannot say who drives whom; conditioning on the receiver's own past can.
- **Quantum mutual information** (`02/10`): the same three faces with von Neumann entropies, $I(A{:}B) = S(A) + S(B) - S(AB) = S(\rho_{AB}\,\|\,\rho_A\otimes\rho_B)$ — Umegaki's relative entropy from `02/09` pointed at a state and its own product, exactly as here. The coins give $1$ bit as a diagonal $\rho_{AB}$ must; the Bell state gives $S(A) + S(B) - S(AB) = 1 + 1 - 0 = 2$ bits — *above today's ceiling*, because the step $H(X|Y) \ge 0$ is the one quantum mechanics breaks. $I(A{:}B)$ is **total** correlation, classical and quantum together; `01/16`'s seed — $\langle Z\otimes Z\rangle \ne \langle Z\rangle\langle Z\rangle$ — is the first symptom of a nonzero $I$.
- **Coupling measures** (`04/12`–`04/13`): when a pair of signals is embedded into a quantum state, $I(A{:}B)$ is one of the two quantum coupling measures (beside the Bures coupling) compared against the classical baselines — and `04/13` shows what it can and cannot see through a given embedding. The plug-in caution of section 4 travels with it.


## Your turn

1. **Three faces on your own table.** Write any $3\times3$ joint table (normalise it), compute $I$ by hand, then $H(X) + H(Y) - H(XY)$ and $H(X) - H(X|Y)$ with `shannon_entropy`; check symmetry by transposing; find the mixture of your table with its own product, $p_t = (1 - t)\,p_X\otimes p_Y + t\,p$, and plot $I(p_t)$ for $t \in [0, 1]$ — is it convex in $t$? (`02/02` §4(b) says what it must be.)
2. **Sign-blind and nonlinear.** Take $X$ uniform on $\{-1, 0, 1\}$ and $Y = X^2$. Compute the Pearson correlation of $X$ and $Y$ (it is $0$) and their mutual information (it is $h(\tfrac13) = 0.918$ bits — $Y$ is a function of $X$, the ceiling of section 2 with $H(Y) < H(X)$). Then send a fair bit through the flip channel at $\epsilon = 0.9$ (a $90\%$-*dis*agreement table) and confirm $I = 1 - h(0.9) = 1 - h(0.1)$. State in one sentence what $I$ measures that correlation does not.
3. **Capacity.** For the flip channel, the input was a fair bit. Let the input be $(p, 1 - p)$ instead and plot $I(X;Y)$ over $p$ for $\epsilon = 0.1$: where is the maximum, and what is its value? Now do the same for the **Z-channel** ($0 \to 0$ always, $1 \to 0$ with probability $\epsilon$): the maximum is no longer at $p = \tfrac12$ — find it numerically for $\epsilon = 0.3$. That maximum is Shannon's channel capacity.
4. **Harder: the bias, and the quantum ceiling.** (a) Repeat section 4 with a $k = 8$ alphabet (uniform independent symbols) for $N \in \{100, 400, 1600\}$ and check the bias against $\frac{(k-1)^2}{2N\ln 2}$; then apply the Miller–Madow correction $\hat I - \frac{(k-1)^2}{2N\ln 2}$ and say what happens to the estimates that fall below zero. (b) Using `von_neumann_entropy` from `qot_course.quantum.density` and `partial_trace` from `qot_course.quantum.composite` (`01/16`, `01/18`), compute $S(A) + S(B) - S(AB)$ for the correlated-coins density matrix $\mathrm{diag}(\tfrac12, 0, 0, \tfrac12)$ and for the Bell state $|\Phi^+\rangle\langle\Phi^+|$. Which one obeys today's ceiling, and which step of section 2's proof fails for the other? (`02/10` makes this the definition.)


## What you built

- You defined mutual information as the relative entropy of a joint table to its own product, computed it term by term on the $90\%$ table ($0.531 = 1 - h(0.1)$ bits), and derived its three faces by hand: $D$ of the joint, $H(X) + H(Y) - H(XY)$, $H(X) - H(X|Y)$.
- You proved $I \ge 0$ (equality iff independent), symmetry, sign-blindness (anti-correlated coins share $1$ bit), and the ceiling $I \le \min(H(X), H(Y))$ — two bits share at most one bit — reached exactly when one variable is a function of the other.
- You sent a fair bit through the flip channel and found $I = 1 - h(\epsilon)$ by hand — Shannon's reason for the quantity — and saw the blend from independence to the coins trace the same curve.
- You defined conditional mutual information, derived the chain rule $I(X;YZ) = I(X;Y) + I(X;Z|Y)$, and from it the data-processing inequality $I(X;Z) \le I(X;Y)$, checked on a chain of two channels and two thousand random ones.
- You measured the upward bias of the plug-in estimator on independent data, matched it to $\frac{(|X|-1)(|Y|-1)}{2N\ln 2}$, and tested an estimate against a shuffled-pairs surrogate null — the rule of every coupling analysis from here on.

The classical-information arc now has its three quantities — entropy, relative entropy, mutual information — each built from the one before. `02/04` adds time, and the quantum lifts begin at `02/09`.


**Dictionary so far** (module 01's table, `02/01`'s entropy row, `02/02`'s relative-entropy row, plus today's row — the classical column is earned here; the quantum column is `02/10`'s promise):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| estimating a probability vector from data: count frequencies, quote $\pm\sqrt{p(1-p)/N}$; a miscalibrated counter biases every frequency by the same factor, and no number of samples fixes it — you calibrate the counter | **state tomography**: estimate $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ from three circuits' counts, each $\pm\sqrt{(1 - \langle\sigma\rangle^2)/N}$, and pack $\hat\rho = \tfrac12(I + \hat{\vec r}\cdot\vec\sigma)$ (project back onto the ball if noise pushes it out); purity, fidelity and trace distance read straight off $\hat\rho$; tomography reports the state *as seen through the meter* — a symmetric misassignment $\epsilon$ shrinks $\hat{\vec r}$ by $1 - 2\epsilon$, a lopsided one also shifts it (calibrate it away — exercise 2; `05/02` meets the same floor on a QPU) — while an idling qubit's *state* really does decohere, coherence as $e^{-t/T_2}$, population as $1 - e^{-t/T_1}$ (channels, `01/19`) |
| **marginal** of a joint table $p(a,b)$: sum over the other variable, $p_A(a) = \sum_b p(a,b)$ (row sums); independent variables factor, $p = p_A \otimes p_B$, and each marginal is the factor; *correlated* variables (the two matched coins) have perfectly random marginals although the pair is not random at all | two systems live on $\mathcal H_A \otimes \mathcal H_B$ (Kronecker product, index $k = 2a + b$, left factor first); a **product state** $\rho_A \otimes \rho_B$ is the independent case; the **partial trace** $\rho_A = \mathrm{tr}_B\,\rho_{AB}$, $(\rho_A)_{aa'} = \sum_b \rho_{ab,a'b}$ (replace each $2\times2$ block by its trace), is the only map that reproduces every local statistic $\mathrm{tr}(\rho_{AB}(M \otimes I)) = \mathrm{tr}(\rho_A M)$; a diagonal $\rho_{AB}$ reduces *exactly* to the row sums; a correlated joint state has **mixed** marginals — the *third* way a state can be mixed: it is part of something larger (`01/18` makes the whole pure and the parts still mixed) |
| **conditional logic** on two bits: controlled-NOT / XOR, $(a, b) \mapsto (a, a \oplus b)$ — a reversible *permutation* of the four outcomes; on a joint probability table it permutes the four entries; two independent operations, one per bit, can never make the bits agree when they did not before | **CNOT**, the $4\times4$ permutation unitary $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes X$ (controlled-$U$ pattern; control = left factor in this course, Qiskit's `cx(1, 0)`); it permutes the four basis vectors and, by linearity, acts on *superpositions* of the control: $\mathrm{CNOT}\,(\alpha|0\rangle + \beta|1\rangle)|0\rangle = \alpha|00\rangle + \beta|11\rangle$ — a product in, a non-product out (amplitude-matrix determinant $\ne 0$); no $M \otimes N$ can do that; CNOT$(H \otimes I)|00\rangle$ is the Bell state, whose corners are *coherence* between $|00\rangle$ and $|11\rangle$ (a product $|{+}{+}\rangle$ has them too; the coins have none) — entanglement proper is `01/18` |
| a joint table whose whole is *certain* (one outcome, probability $1$) has certain marginals; mixed marginals force a mixed whole (the coins); the correlation of two bits is fully described by their joint table, and any table is a mixture of point masses — every classical correlation is a *shared randomness* | **entanglement**: a *pure* joint state that is not a product (amplitude-matrix $\det\Psi \ne 0$) has **mixed parts** — pure whole, mixed parts, impossible for any table; measured by the **entanglement entropy** $S(\rho_A) = -\sum_i \lambda_i\log_2\lambda_i$ (eigenvalues of the part; $= S(\rho_B)$; $1$ bit for a Bell state; invariant under local gates $U\otimes V$) — **valid only when the whole is pure**: the coins have $S(\rho_A) = 1$ bit and no entanglement; a mixed state is **separable** if it is a mixture of products, and for two qubits the **partial transpose** decides it (a negative eigenvalue = entangled: Bell $-\tfrac12$, coins none); Bell and coins share every $Z$-statistic and differ in $X$: $\langle X\otimes X\rangle = 1$ vs $0$ |
| the general evolution of a probability vector is a **stochastic matrix** (Markov kernel) $p \to Tp$: columns are probability vectors, so probabilities stay non-negative and sum to $1$; a permutation is the reversible case; noise = a coin you did not see being tossed | a **quantum channel**: the general evolution $\rho \to \sum_k K_k\rho K_k^\dagger$ (Kraus operators, $\sum_k K_k^\dagger K_k = I$) — *derived* as "couple to an environment, evolve unitarily, trace the environment out" ($K_k = \langle k|_E\,U\,|0\rangle_E$); linear, trace-preserving and **completely** positive (the transpose is positive but sends half a Bell pair to a matrix with a $-\tfrac12$ eigenvalue — not a channel); a unitary is the reversible case ($K = U$), a stochastic matrix is the diagonal case ($K_{ij} = \sqrt{T_{ij}}\,|i\rangle\langle j|$); **decoherence = entanglement with an environment, then a partial trace**: dephasing (the environment learns the bit; coherence $\times e^{-t/T_\varphi}$), amplitude damping (energy leaks out; population $\times e^{-t/T_1}$, coherence $\times e^{-t/2T_1}$), depolarizing ($\rho \to (1-p)\rho + p\,I/2$); together they give $1/T_2 = 1/2T_1 + 1/T_\varphi$, hence $T_2 \le 2T_1$ — the laws `01/15` fitted |
| **Shannon entropy** $H(p) = -\sum_x p_x\log_2 p_x$: the average *surprise* $-\log_2 p_x$ (the log is forced by additivity of independent surprises); $0$ for a certain outcome, $\log_2 n$ for the uniform (Jensen), concave; **binary entropy** $h(p)$; joint and conditional entropy with the chain rule $H(AB) = H(A) + H(B|A)$, $H(B|A) \ge 0$, hence $H(A) \le H(AB)$ — a part is never more uncertain than the whole (coins: $1 = 1$; independent: $H(AB) = H(A) + H(B)$); the floor of lossless compression (Shannon 1948) | **von Neumann entropy** $S(\rho) = -\mathrm{tr}\,\rho\log_2\rho = H(\text{eigenvalues of }\rho)$ — the Shannon entropy of the eigenvalue coin (`01/13`): $0$ for a pure state, $1$ bit for $I/2$, $h\big(\tfrac{1 + |\vec r|}{2}\big)$ for a qubit; *but* the chain rule's positivity fails: the Bell state has $S(AB) = 0 < S(A) = 1$ (`01/18`) — a negative conditional entropy, `02/11`; the quantum side is earned in `02/09` |
| **relative entropy** (Kullback–Leibler) $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ — the *extra* bits per symbol of coding with model $q$ when the truth is $p$ (cross-entropy minus entropy); **Gibbs**: $D \ge 0$, $= 0$ iff $p = q$ (from $\ln x \le x - 1$) — which re-proves $H \le \log_2 n$ and the source-coding floor; **not** a distance: asymmetric, no triangle inequality, $+\infty$ when $q$ gives zero to something $p$ can do; additive on products, jointly convex, and **monotone under any stochastic map** $D(Tp\|Tq) \le D(p\|q)$ — processing cannot create distinguishability; $D(p\,\|\,\text{uniform}) = \log_2 n - H(p)$ | **Umegaki relative entropy** $S(\rho\|\sigma) = \mathrm{tr}\,\rho(\log_2\rho - \log_2\sigma)$ (`02/09`): equals $D$ when $\rho, \sigma$ are diagonal in one basis, $+\infty$ when the support of $\rho$ leaves that of $\sigma$, $\ge 0$ (Klein), monotone under every quantum channel (Lindblad 1975) — and it *sees the corners*: $|{+}\rangle$ vs $I/2$ have identical diagonals, classical $D = 0$, quantum $S = 1$ bit |
| **mutual information** $I(X;Y) = D(p_{XY}\,\|\,p_X \otimes p_Y) = H(X) + H(Y) - H(XY) = H(X) - H(X|Y)$ — the bits the joint saves over the product, the failure of independence; symmetric, $\ge 0$ (Gibbs), $= 0$ iff independent, sign-blind (anti-correlation counts), sees nonlinear dependence; **ceiling** $I \le \min(H(X), H(Y))$ — two bits share at most $1$ bit; $1 - h(\epsilon)$ through a symmetric flip channel; **conditional** $I(X;Y|Z)$, chain rule $I(X;YZ) = I(X;Y) + I(X;Z|Y)$, hence the **data-processing inequality** $I(X;Z) \le I(X;Y)$ on a chain $X \to Y \to Z$; estimated from counts it is biased *upward* by $\approx \frac{(|X|-1)(|Y|-1)}{2N\ln 2}$ — compare with surrogates | **quantum mutual information** $I(A{:}B) = S(A) + S(B) - S(AB) = S(\rho_{AB}\,\|\,\rho_A \otimes \rho_B)$ (`02/10`): the same formula with von Neumann entropies, $\ge 0$, $= 0$ iff product; *total* correlation, classical and quantum together — the coins give $1$ bit (as here), the Bell state $2$ bits, **above** the classical ceiling, because $S(AB) = 0 < S(A)$; monotone under local channels |
| transport plan | quantum coupling |

## What's next

Mutual information is symmetric: it says *that* two signals are related, not *which leads which*. `02/04_transfer_entropy` adds time — condition on the receiver's own past and ask what the sender's past adds: $I(X_{t+1}; Y_t\,|\,X_t)$, today's conditional mutual information — and closes the classical-information arc with the estimation discipline of section 4 applied to signals.


## References

- C. E. Shannon, *A mathematical theory of communication*, Bell Syst. Tech. J. **27**, 379–423 and 623–656 (1948) — Part II §12 (equivocation and the "rate of transmission" $H(X) - H(X|Y)$), §13–14 (the fundamental theorem, capacity). doi:10.1002/j.1538-7305.1948.tb01338.x
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §2.3–2.5 (mutual information, its three forms, conditional mutual information, chain rule), Thm 2.6.3 (non-negativity), §2.8 (data-processing inequality), §7.1.4 (binary symmetric channel), Problem 7.8 (Z-channel), §7.1 (capacity). doi:10.1002/047174882X
- G. A. Miller, *Note on the bias of information estimates*, in *Information Theory in Psychology* (H. Quastler, ed.), Free Press (1955), 95–100 — the $\frac{(k-1)}{2N}$ bias of plug-in entropies, hence the $\frac{(|X|-1)(|Y|-1)}{2N}$ bias of plug-in mutual information.
- L. Paninski, *Estimation of entropy and mutual information*, Neural Computation **15**, 1191–1253 (2003) — the modern treatment of plug-in bias and its corrections. doi:10.1162/089976603321780272
- T. Schreiber, *Measuring information transfer*, Phys. Rev. Lett. **85**, 461–464 (2000) — transfer entropy as a conditional mutual information, `02/04`. doi:10.1103/PhysRevLett.85.461
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, Cambridge University Press (2010) — §11.3.2 (quantum mutual information, definition), §11.3.4 (subadditivity, i.e. $I \ge 0$), `02/10`. doi:10.1017/CBO9780511976667

**Previous:** `notebooks/02_InformationAndGeometry/02_kl_divergence.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/04_transfer_entropy.ipynb`
